# 17 | Fixed-Optimiser-Step Training-Data Comparison

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Research question:** Does increasing Train windows from 500 to 1,000 per person improve Weighted CNN when optimiser updates are identical? Reuse exactly the nested 29-person HARTH development cohort and the three participant-held-out GroupKFold assignments. The protected original Test IDs S023 and S032 are never loaded. This is a development-only analysis.

## 01. Imports and reusable functions

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, balanced_accuracy_score

# Resolve project root and shared Python modules.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root: raise RuntimeError("Open notebook within repository.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, prepare_harth_nested_window_cohort
from core.temporal_cnn import train_temporal_cnn_fixed_steps
from core.ml_baselines import evaluate_predictions_by_subject, classification_class_report
from core.group_cross_validation import compare_models_by_subject
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())

## 02. Prepare nested development windows

Replicate Stage 16: the original base-500 windows are fully contained in the nested cap-1000 samples. All sampling is chronological and independent of activity labels.

In [ ]:
# Keep the original reserved Test participants out of all loading and fitting.
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
X, metadata, overview = prepare_harth_nested_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS, data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES, base_cap=500, expanded_cap=1000,
    window_size=100, step_size=50, sampling_rate=50,
)
base_mask = metadata["in_base_500"].to_numpy(dtype=bool)
base_rows = np.flatnonzero(base_mask)
labels = metadata["label"].to_numpy()
subjects = metadata["subject_id"].to_numpy()
assert len(base_rows) == 14201 and len(X) == 28201
assert not (set(subjects) & RESERVED_TEST)
print("Base / expanded windows:", len(base_rows), len(X))
display(overview)

## 03. Freeze identical subject-disjoint folds

Split only base-500 subject rows. Both training-size variants use identical validation people and identical base windows; newly added samples can only come from Train people.

In [ ]:
# Match Stage 16 GroupKFold group assignment.
FOLDS = 3
group_folds = list(GroupKFold(n_splits=FOLDS).split(
    np.zeros(len(base_rows)), labels[base_rows], groups=subjects[base_rows]
))
fold_design = []
for fold, (tr, va) in enumerate(group_folds, start=1):
    train_people = set(subjects[base_rows[tr]])
    validation_people = set(subjects[base_rows[va]])
    assert not (train_people & validation_people)
    expanded_train = np.flatnonzero(np.isin(subjects, list(train_people)))
    assert np.isin(base_rows[tr], expanded_train).all()
    fold_design.append({
        "fold": fold, "train_people": len(train_people),
        "validation_people": len(validation_people),
        "train_500": len(tr), "train_1000": len(expanded_train),
        "fixed_validation": len(va),
    })
display(pd.DataFrame(fold_design))

## 04. Train weighted CNNs for exactly equal optimiser steps

Each model is freshly initialised with the same seed, Adam settings and batch size. Both see exactly **400 optimiser updates** and report Validation at predetermined steps 100, 200, 300 and 400. Validation is not used to choose a checkpoint; the 400-step prediction is the reported result. Equal updates do not imply an equal number of exposures per source window.

In [ ]:
# Run two model variants in each of the same three folds.
MAX_STEPS = 400
EVAL_EVERY = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
fold_records, people_frames, class_frames, histories = [], [], [], []
for fold, (train_local, validation_local) in enumerate(group_folds, start=1):
    train_rows_500 = base_rows[train_local]
    val_rows = base_rows[validation_local]
    train_people = set(subjects[train_rows_500])
    train_rows_1000 = np.flatnonzero(np.isin(subjects, list(train_people)))
    y_val = labels[val_rows]
    for name, selected_train in [
        ("Weighted CNN 500", train_rows_500),
        ("Weighted CNN 1000", train_rows_1000),
    ]:
        print(f"Fold {fold}/{FOLDS} | {name} | {len(selected_train)} Train windows")
        model, history, predicted, statistics = train_temporal_cnn_fixed_steps(
            X_train=X[selected_train], y_train=labels[selected_train],
            X_validation=X[val_rows], y_validation=y_val,
            classes=TASK_CLASSES, max_steps=MAX_STEPS,
            eval_every=EVAL_EVERY, batch_size=128, learning_rate=1e-3,
            seed=42, device=DEVICE, class_weighting="inverse_sqrt",
        )
        fold_records.append({
            "fold":fold, "model":name, "train_windows":len(selected_train),
            "validation_windows":len(val_rows),
            "macro_f1":f1_score(y_val,predicted,labels=TASK_CLASSES,
                                average="macro",zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(y_val,predicted),
        })
        frame = evaluate_predictions_by_subject(
            y_val,predicted,subjects[val_rows],TASK_CLASSES,name
        )
        frame.insert(0,"fold",fold)
        people_frames.append(frame)
        report=classification_class_report(
            y_val,predicted,classes=TASK_CLASSES,
            class_names=[ACTIVITY_NAMES[code] for code in TASK_CLASSES],
        )
        report["fold"],report["model"]=fold,name
        class_frames.append(report)
        history["fold"],history["model"]=fold,name
        histories.append(history)
fold_results=pd.DataFrame(fold_records)
subject_results=pd.concat(people_frames,ignore_index=True)
class_results=pd.concat(class_frames,ignore_index=True)
history_table=pd.concat(histories,ignore_index=True)
display(fold_results.round(4))

## 05. Compare final-step Macro F1 across folds

Fixed training-step results can be compared with each other, not directly interpreted as an independent confirmation of a causal effect. Stage 16 used fixed epochs instead, so its training budget differs.

In [ ]:
# Display mean and standard deviation across the three development folds.
summary=fold_results.groupby("model",as_index=False).agg(
    macro_f1_mean=("macro_f1","mean"),
    macro_f1_sd=("macro_f1","std"),
    balanced_accuracy_mean=("balanced_accuracy","mean"),
)
display(summary.round(4))
fig,ax=plt.subplots(figsize=(9,4))
for name,group in fold_results.groupby("model",sort=False):
    ax.plot(group["fold"],group["macro_f1"],marker="o",label=name)
ax.set(title="Matched 400-update Weighted CNN comparison",
       xlabel="Subject-held-out fold",ylabel="Macro F1",
       xticks=[1,2,3],ylim=(0,1))
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 06. Review checkpoint trajectories without cherry-picking

Intermediate Validation curves are diagnostics. The predeclared final step 400 is used for model comparison, even if an earlier point achieves a higher score.

In [ ]:
# Show both training trajectories, separated by fold.
fig,axes=plt.subplots(3,1,figsize=(10,10),sharex=True)
for fold,ax in zip(range(1,FOLDS+1),axes):
    for name,group in history_table.loc[
        history_table["fold"]==fold
    ].groupby("model",sort=False):
        ax.plot(group["step"],group["validation_macro_f1"],
                marker="o",label=name)
    ax.set(title=f"Fold {fold}",ylabel="Validation Macro F1",ylim=(0,1))
    ax.grid(alpha=0.2)
    ax.legend()
axes[-1].set_xlabel("Optimizer updates")
fig.tight_layout()
plt.show()

## 07. Inspect participant and class-level outcomes

The same independent participants are validated under both conditions. Shuffling and stair performance require separate inspection because average metrics can obscure difficult classes.

In [ ]:
# Pair fixed-task Macro F1 from identical held-out people.
paired,paired_summary=compare_models_by_subject(
    subject_results,"Weighted CNN 1000","Weighted CNN 500",
    score_column="macro_f1_fixed_task",
)
display(pd.DataFrame([paired_summary]).round(4))
display(paired.round(4))
fig,ax=plt.subplots(figsize=(12,4))
ax.bar(paired["subject_id"],paired["difference_a_minus_b"])
ax.axhline(0,color="black",linewidth=0.8)
ax.set(title="Participant-level difference: 1000 minus 500",
       xlabel="Held-out development participant",ylabel="Fixed-task Macro F1 difference")
ax.tick_params(axis="x",rotation=65)
ax.grid(axis="y",alpha=0.2)
fig.tight_layout()
plt.show()

# Report minority-class F1 per fold, not invented pooled metrics.
display(class_results.loc[
    class_results["class"].isin(
        ["Shuffling","Stairs ascending","Stairs descending"]
    ),["fold","model","class","support","precision","recall","f1"]
].round(4))

## 08. Scientific quality gate

This experiment only controls the optimiser-update count. Two Training sizes still imply different subject-window reuse and effective epoch counts. The original Test people remain excluded from loading, training and evaluation.

In [ ]:
# Confirm fixed steps, groups and paired subject coverage.
assert len(fold_results)==2*FOLDS
assert len(subject_results)==2*len(DEVELOPMENT_SUBJECTS)
assert len(paired)==len(DEVELOPMENT_SUBJECTS)
assert set(history_table.groupby(["fold","model"])["step"].max())=={MAX_STEPS}
assert not (set(subject_results["subject_id"]) & RESERVED_TEST)
assert subject_results.groupby(["model","subject_id"]).size().eq(1).all()
assert fold_results["macro_f1"].between(0,1).all()
print("Equal-optimizer-step development comparison: PASSED")
print("Protected original Test participants used: NO")
print("Independent generalisation established: NO")